In [2]:
# ────────────────────────────────────────────────
# CELL 1 — Setup SparkSession
# ────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
import os
spark = SparkSession.builder \
    .appName("LaLiga_Bronze_to_Silver_Teams") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [3]:
spark

In [4]:
BRONZE_DIR = "../data/1_bronze/teams"
SILVER_DIR = "../data/2_silver/teams"

In [5]:
# Daftar 6 kategori
CATEGORIES = [
    "teams_attacking",
    "teams_defending",
    "teams_passing",
    "teams_pressing",
    "teams_sequences",
    "teams_misc",
]

In [ ]:
print(f"✅ SparkSession aktif: {spark.sparkContext.appName}")
print(f"📂 Bronze: {BRONZE_DIR}")
print(f"📂 Silver: {SILVER_DIR}")
print(f"📋 Kategori: {len(CATEGORIES)}")

In [ ]:
# ────────────────────────────────────────────────
# CELL 2 — Load Semua Bronze JSON
# ────────────────────────────────────────────────

bronze_dfs = {}
for cat in CATEGORIES:
    path = f"{BRONZE_DIR}/{cat}.json"
    df = spark.read.json(path, multiLine=True)
    bronze_dfs[cat] = df
    print(f"  📄 {cat:25s} → {df.count()} baris, {len(df.columns)} kolom")
print(f"\n✅ {len(bronze_dfs)} file berhasil di-load")

  📄 teams_attacking           → 20 baris, 9 kolom
  📄 teams_defending           → 20 baris, 10 kolom
  📄 teams_passing             → 20 baris, 17 kolom
  📄 teams_pressing            → 20 baris, 9 kolom
  📄 teams_sequences           → 20 baris, 10 kolom
  📄 teams_misc                → 20 baris, 13 kolom

✅ 6 file berhasil di-load


In [ ]:
# path = f"../data/1_bronze/teams/teams_attacking.json"

# df = spark.read.json(path, multiLine=True)
# from pyspark.sql.types import StructField
# df.columns

# for field in df.schema.fields:
#     print(field)

# StructField()

In [22]:
# ────────────────────────────────────────────────
# CELL 3 — Definisikan Fungsi Cleaning
# ────────────────────────────────────────────────

# 3a. Parse kolom string '%' → float (desimal)
def parse_pct_columns(df):
    """
    Convert kolom string '%' → float.
    Contoh: '13.73%' → 0.1373
    Hanya proses kolom bertipe string selain 'club'.
    """
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string" and field.name != "club":
            df = df.withColumn(
                field.name,
                (
                    F.regexp_replace(F.col(field.name), "%", "")
                    .cast(FloatType()) / 100
                )
            )
    return df



In [23]:
# 3b. Mapping nama klub: standardisasi dari berbagai variasi
CLUB_NAME_MAPPING = {
    "Atlético Madrid": "Atletico Madrid",
    "Atlético": "Atletico Madrid",
    "Athletic Bilbao": "Athletic Club",
    "Athletic": "Athletic Club",
    "Celta Vigo": "Celta Vigo",
    "Celta": "Celta Vigo",
    "Real Betis": "Real Betis",
    "Betis": "Real Betis",
    "Real Oviedo": "Real Oviedo",
    "Oviedo": "Real Oviedo",
    "Rayo Vallecano": "Rayo Vallecano",
    "Rayo": "Rayo Vallecano",
    "Espanyol": "Espanyol",
    "Espanyol": "Espanyol",
    "Alaves": "Deportivo Alaves",
    "Alavés": "Deportivo Alaves",
}
def standardize_club_names(df):
    """
    Standardisasi nama klub menggunakan mapping.
    Nama yang tidak ada di mapping tetap tidak berubah.
    """
    mapping_expr = F.col("club")
    for old_name, new_name in CLUB_NAME_MAPPING.items():
        mapping_expr = F.when(
            F.col("club") == old_name, F.lit(new_name)
        ).otherwise(mapping_expr)
    
    df = df.withColumn("club", mapping_expr)
    return df

In [24]:
# 3c. Gabungkan semua cleaning steps
def clean_bronze_to_silver(df):
    """Pipeline cleaning: parse % → standardisasi nama."""
    df = parse_pct_columns(df)
    df = standardize_club_names(df)
    return df
print("✅ Fungsi cleaning sudah didefinisikan:")
print("   - parse_pct_columns()")
print("   - standardize_club_names()")
print("   - clean_bronze_to_silver()")

✅ Fungsi cleaning sudah didefinisikan:
   - parse_pct_columns()
   - standardize_club_names()
   - clean_bronze_to_silver()


In [ ]:
# ────────────────────────────────────────────────
# CELL 4 — Process Semua Kategori (Bronze → Silver)
# ────────────────────────────────────────────────
silver_dfs = {}
for cat in CATEGORIES:
    print(f"\n{'─'*50}")
    print(f"  🔄 Processing: {cat}")
    print(f"{'─'*50}")
    
    df_bronze = bronze_dfs[cat]
    
    # Sebelum cleaning
    print(f"  BEFORE:")
    print(f"    Schema:")
    df_bronze.printSchema()
    
    # Apply cleaning
    df_silver = clean_bronze_to_silver(df_bronze)
    
    # Sesudah cleaning
    print(f"  AFTER:")
    df_silver.printSchema()
    df_silver.show(3, truncate=False)
    
    silver_dfs[cat] = df_silver
print(f"\n✅ {len(silver_dfs)} kategori berhasil di-clean")

In [ ]:
# ────────────────────────────────────────────────
# CELL 5 — Validasi Silver Data
# ────────────────────────────────────────────────
print("🔍 Validasi Silver Data:\n")
for cat, df in silver_dfs.items():
    count = df.count()
    n_cols = len(df.columns)
    n_clubs = df.select("club").distinct().count()
    
    # Cek null
    null_count = 0
    for col_name in df.columns:
        null_count += df.filter(F.col(col_name).isNull()).count()
    
    # Cek tidak ada lagi string '%'
    str_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string" and f.name != "club"]
    
    status = "✅" if count == 20 and null_count == 0 and len(str_cols) == 0 else "⚠️"
    
    print(f"  {status} {cat:25s} → {count} baris, {n_cols} kolom, {n_clubs} klub unik, {null_count} nulls, sisa string: {str_cols}")
# Cek nama klub sudah ter-standardisasi
print(f"\n📋 Daftar nama klub setelah standardisasi:")
silver_dfs["teams_attacking"].select("club").distinct().orderBy("club").show(20, truncate=False)

🔍 Validasi Silver Data:

  ✅ teams_attacking           → 20 baris, 9 kolom, 20 klub unik, 0 nulls, sisa string: []
  ✅ teams_defending           → 20 baris, 10 kolom, 20 klub unik, 0 nulls, sisa string: []
  ✅ teams_passing             → 20 baris, 17 kolom, 20 klub unik, 0 nulls, sisa string: []
  ✅ teams_pressing            → 20 baris, 9 kolom, 20 klub unik, 0 nulls, sisa string: []
  ✅ teams_sequences           → 20 baris, 10 kolom, 20 klub unik, 0 nulls, sisa string: []
  ✅ teams_misc                → 20 baris, 13 kolom, 20 klub unik, 0 nulls, sisa string: []

📋 Daftar nama klub setelah standardisasi:
+----------------+
|club            |
+----------------+
|Athletic Club   |
|Atletico Madrid |
|Barcelona       |
|Celta Vigo      |
|Deportivo Alaves|
|Elche           |
|Espanyol        |
|Getafe          |
|Girona          |
|Levante         |
|Mallorca        |
|Osasuna         |
|Rayo Vallecano  |
|Real Betis      |
|Real Madrid     |
|Real Oviedo     |
|Real Sociedad   |
|Sevilla

26/07/09 21:15:24 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 499833 ms exceeds timeout 120000 ms
26/07/09 21:15:24 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/09 21:15:30 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1530)
	at o

In [ ]:
# spark.stop()

NameError: name 'spark' is not defined